# Installation
install torch for you GPU/MPS/CPU https://pytorch.org/get-started/locally/



In [ ]:
%load_ext autoreload
%autoreload 2
try:
    import deepali
    import TPTBox
except Exception:
    %pip install TPTBox ruamel.yaml configargparse
    %pip install hf-deepali
    %pip install nnunetv2

    import deepali
    import TPTBox
from pathlib import Path
from typing import Literal

import torch
from TPTBox import NII, POI_Global, to_nii
from TPTBox.core.vert_constants import Full_Body_Instance
from treg.angle import compute_angles
import pandas as pd

In [ ]:
from TPTBox.core.dicom.dicom_extract import extract_dicom_folder

dataset = "/media/data/robert/code/TReg/private/dataset-treg"
extract_dicom_folder(Path("/media/data/robert/code/TReg/private/Pat0001"), Path(dataset))

In [ ]:




from TPTBox.stitching import stitching
from TPTBox.segmentation import run_vibeseg
from TPTBox import BIDS_Global_info,BIDS_FILE
from TPTBox.core.vert_constants import Full_Body_Instance
import numpy as np
bgi = BIDS_Global_info(dataset)
dataset_id = 11

subs = {}

for sub,subj in bgi.iter_subjects(sort=True):
    q = subj.new_query(flatten=True)
    q.filter_format("ct")
    q.filter_filetype("nii.gz")
    #q.filter("acq","iso",required=False) # TODO always ISO images avalible?
    q.filter("part",lambda x: x != "localizer",required=False)
    ##
    img:dict[str,list[BIDS_FILE]] = {}
    segs:dict[str,list]  = {}
    for file in q.loop_list(sort=True):
        out_file = file.get_changed_path("nii.gz","msk",info={"seg":f"VIBESeg-{dataset_id}"})
        out = run_vibeseg(file, out_file, gpu=0, ddevice="cuda", dataset_id=dataset_id)
        j:dict = file.open_json()
        # split by different features
        kernel = j.get("ConvolutionKernel","ct")
        kernel += "-"+str(file.get("ses",""))
        kernel += "-"+str(file.get("acq",""))
        kernel += "-"+str(file.get("part",""))
        if kernel not in img:
            img[kernel] = []
            segs[kernel] = []
        img[kernel].append(file)
        segs[kernel].append(out)
    for name, l in img.items():
        if len(l) <= 1:
            continue
        seq = "-".join(sorted([str(a.get("sequ")) for a in l]))
        out = l[0].get_changed_path("nii.gz","msk","rawdata",info={"sequ":f"stiched-{seq}"})
        out_seg = l[0].get_changed_path("nii.gz","msk",info={"sequ":f"stiched-{seq}","seg":f"VIBESeg-{dataset_id}"})
        out_ramp = l[0].get_changed_path("nii.gz","ramp",info={"sequ":f"stiched-{seq}","seg":f"VIBESeg-{dataset_id}"},non_strict_mode=True)
        if not out.exists():
            stitching(l,out,is_ct=True,verbose=True,verbose_stitching=True,dtype=np.int16,store_ramp=True,ramp_path=out_ramp)
        if not out_seg.exists():
            stitching(segs[name],out_seg,is_seg=True,verbose=True,verbose_stitching=True)
            to_nii(out_seg,True).set_dtype("smallest_uint").save(out_seg)
        if sub not in subs or (l[0].get("acq","")=="iso" and subs[sub]["acq"] != "iso") or (l[0].get("acq","")=="ax" and subs[sub]["acq"] not in ["iso","ax"]):
            u = to_nii(out_seg,True).unique()
            if all(a in u for a in range(11,15)):# TDDO version 12 add test for 100
                subs[sub] = {"img":out,"seg":out_seg,"dataset":l[0].dataset,"bin_msk":out_ramp,"acq":l[0].get("acq","")}

In [ ]:
subject_name = None

if subject_name is None:
    subject_name = next(iter(subs.keys()))
print(subject_name)

In [ ]:
#############################################
################# Input #####################
#############################################

files = subs[subject_name]
#{"img":out,"seg":out_seg,"dataset":l[0].dataset}
ds = BIDS_FILE(files["img"],files["dataset"])
sides = ["left","right"]
for side in sides:
    files[side] = {}
    files[side]["target_out_poi"] = ds.get_changed_path("json","poi",info={"desc":"atlas","seg":side},additional_folder=side)
    files[side]["target_out_subdivided"] = ds.get_changed_path("nii.gz","msk",info={"desc":"atlas","seg":side},additional_folder=side)
    files[side]["target_out_angle"] = ds.get_changed_path("nii.gz","msk",info={"desc":"angle","seg":side},additional_folder=side)
    files[side]["target_out_angle2"] = ds.get_changed_path("nii.gz","msk",info={"desc":"angle-veerman","seg":side},additional_folder=side)
    files[side]["mirror"] ="right" in side
    
    

# Atlas
atlas_seg_file: str | Path = "data/sub-atlas_seg-VIBESeg-12_msk.nii.gz"  # default is a left leg
atlas_file: str | Path = "data/sub-atlas_seg-poi_poi.json"
atlas_seg_subdivided_file: str | Path | None = "data/sub-atlas_seg-subregion_msk.nii.gz"

# Device settings

ddevice: Literal["cpu", "cuda", "mps"] = "cuda"
gpu = 0  # Only used for cuda


In [ ]:
def resolve_device(
    ddevice: Literal["cpu", "cuda", "mps"],
    gpu: int = 0,
) -> torch.device:
    if ddevice == "cuda":
        if torch.cuda.is_available():
            torch.cuda.set_device(gpu)
            return torch.device(f"cuda:{gpu}")
        else:
            print("⚠️ CUDA requested but not available → falling back to CPU")

    if ddevice == "mps":
        if torch.backends.mps.is_available() and torch.backends.mps.is_built():
            return torch.device("mps")
        else:
            print("⚠️ MPS requested but not available → falling back to CPU")

    return torch.device("cpu")


device = resolve_device(ddevice, gpu)
print(f"✅ Using device: {device}")

In [ ]:
# Parameters
lr: float = 0.001
max_steps: int = 1500
min_delta: float = 0.000001
pyramid_levels: int = 4
coarsest_level: int = 3
finest_level: int = 0
weights: dict = {"be": 0.00001, "seg": 1, "Dice": 0.01, "Tether": 0.001}


# IDs from the segmentation model, for you own segmentation update it with the label ID you want to use.
leg_ids = [
    Full_Body_Instance.femur_left,  # <- you can use integer for the labels here.
    Full_Body_Instance.patella_left,
    Full_Body_Instance.tibia_left,
    Full_Body_Instance.fibula_left,
]
# Mirrors the ids for left and right legs. Update for you own. If not needed, make an empty dict.
mapping_mirror = {
    Full_Body_Instance.femur_right.value: Full_Body_Instance.femur_left.value,
    Full_Body_Instance.patella_right.value: Full_Body_Instance.patella_left.value,
    Full_Body_Instance.tibia_right.value: Full_Body_Instance.tibia_left.value,
    Full_Body_Instance.fibula_right.value: Full_Body_Instance.fibula_left.value,
    Full_Body_Instance.femur_left.value: Full_Body_Instance.femur_right.value,
    Full_Body_Instance.patella_left.value: Full_Body_Instance.patella_right.value,
    Full_Body_Instance.tibia_left.value: Full_Body_Instance.tibia_right.value,
    Full_Body_Instance.fibula_left.value: Full_Body_Instance.fibula_right.value,
}

## Generate Segmentation


In [ ]:
nii = to_nii(files["seg"],True)
while nii.max()<=100:
    print(nii.unique(), "requires manual split")
    bf = BIDS_FILE(files["seg"],files["dataset"])
    mask = bf.get_changed_path(parent=bf.parent,info={"seg":"left-right-mask"})
    if mask.exists():
        r = to_nii(mask,True).resample_from_to(nii)
        for i in range(13, 25):
            nii[np.logical_and(nii == i, r == 1)] += 100
        nii.save(files["seg"])
    else:
        (nii*0).rescale((4,4,4)).save(mask)
        input("paint the left side with 1")

In [ ]:
bin_msk = to_nii(files["bin_msk"],True)
if len(bin_msk.shape) == 4:
    print(bin_msk.shape)
    bin_msk = bin_msk.set_array(bin_msk.get_array().sum(-1)).clamp(0,1)
    bin_msk.set_dtype_("smallest_uint").save(files["bin_msk"])
    print(bin_msk.shape)

In [ ]:
# assert not mirror
from TPTBox.registration import Template_Registration
bin_msk = to_nii(files["bin_msk"],True)
if len(bin_msk.shape) == 4:
    bin_msk.set_array_(bin_msk.get_array().sum(-1))
for side in sides:

    files[side]["target_out_poi"].parent.mkdir(exist_ok=True)
    # load
    moving_img = to_nii(atlas_seg_file, True)
    target = to_nii(files["seg"], True)

    # change label for mirroring
    mirror = files[side]["mirror"]
    if mirror:
        target = target.map_labels(mapping_mirror)
    # Limit to only used labels
    print(target.unique(), moving_img.unique())
    seg = target.extract_label(leg_ids, True)
    print("unique", seg.unique(), moving_img.unique())

    # Run Template_Registration
    reg = Template_Registration(
        seg,  # Target segmentation
        moving_img.extract_label(leg_ids, True),  # Starting Atlas Segmentation (not the split one)
        same_side=not mirror,
        lr=lr,
        max_steps=max_steps,
        min_delta=min_delta,
        pyramid_levels=pyramid_levels,
        coarsest_level=coarsest_level,
        finest_level=finest_level,
        # loss_terms=loss_terms,
        # poi_target_cms=None,
        # poi_cms=poi_atlas_cms,  # Can be None, than it will be computed automatically
        weights=weights,
        gpu=0,
        ddevice=ddevice,
        fixed_mask=bin_msk,
    )
    # Transfer atlas to target
    print("Transfer atlas to target")
    poi_in = POI_Global.load(atlas_file)
    atlas_reg = reg.transform_poi(poi_in)  # Transferring the atlas points
    atlas_reg.info = poi_in.info
    atlas_reg.to_global().save_mrk(files[side]["target_out_poi"])
    atlas_reg.save(files[side]["target_out_poi"])

    if atlas_seg_subdivided_file is not None:
        n = reg.transform_nii(
            to_nii(atlas_seg_subdivided_file, True), allow_only_same_grid_as_moving=False
        )  # Transferring the atlas subdivisions
        s = seg.extract_label([13, 15, 113, 115], keep_label=True) % 100
        s.map_labels_({13: 7, 15: 8})
        n = n.infect_(s) * (seg + (1 - bin_msk).dilate_msk(2)).clamp(0, 1)
        s = s.erode_msk_euclid(3)
        n[s != 0] = s[s != 0]
        n.save(files[side]["target_out_subdivided"])

In [ ]:
for side in sides:
    atlas_reg = POI_Global.load(files[side]["target_out_poi"])
    angle_dict, _, _ = compute_angles(atlas_reg.to_global(), files[side]["target_out_angle"])
    print(dict)
    df = pd.DataFrame([angle_dict])
    df.to_excel(str(files[side]["target_out_angle"]).split(".")[0] + ".xlsx")

In [ ]:
from treg.veerman_rules_based import run_single_case
for side in sides:
    poi = run_single_case(
        nii=files[side]["target_out_subdivided"],
        stl_folder=Path(files[side]["target_out_subdivided"]).parent / "stl",
        side="L" if side.lower() == "left" else "R",
        output_csv=str(files[side]["target_out_angle2"]).split(".")[0] + ".csv",
    )
#poi.save_mrk(files[side]["target_out_angle2"])